In [4]:
import pandas as pd

df = pd.read_json("/content/iam_anomaly_2026.jsonl", lines=True)

ValueError: Unexpected character found when decoding object value

In [ ]:
print("Shape:", df.shape)

In [ ]:
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.astype(str).duplicated().sum())
#astype(str) is added as dataset is in json form

In [ ]:
df.info()

In [ ]:
df['event_type'].value_counts().head(15)
#event_type describes what action or activity occurred in the authentication system.

In [ ]:
df['category'].value_counts()

In [ ]:
failed_df = df[df['event_type'].isin([
    'auth_access_log_failed',
    'auth_login_failed_rejected'
])].copy()

print("Failed login records:", len(failed_df))

In [ ]:
failed_df = failed_df.set_index('timestamp')

login_counts = failed_df.resample('5min').size()

print(login_counts.head())

In [ ]:
import matplotlib.pyplot as plt

print("Matplotlib imported successfully")
print(type(login_counts))
print(login_counts.shape)

In [ ]:
plt.figure(figsize=(12, 5))

plt.scatter(login_counts.index, login_counts.values, s=10)

plt.xlabel("Time")
plt.ylabel("Failed Login Attempts")
plt.title("Failed Login Attempts Over Time")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print(login_counts.sort_values(ascending=False).head(10))
#10 failded login datapoints

In [ ]:
model_df = login_counts.to_frame(name='failed_login_count')

print(model_df.head())
#This gives us one column containing failed login counts for every 5-minute interval.

In [ ]:
model_df['hour'] = model_df.index.hour
model_df['day'] = model_df.index.dayofweek
print(model_df.head())
#we are extracting hours and data count from timestamp

In [ ]:
model_df['previous_login_count'] = model_df['failed_login_count'].shift(1)

print(model_df.head())
#getting login failded  for 5 min interval
#shift(1) is for 1 line shift at starting

In [ ]:
model_df['rolling_avg'] = (
    model_df['failed_login_count'].shift(1).rolling(window=3).mean()
)

print(model_df.head(10))

In [ ]:
model_df = model_df.dropna()

print(model_df.isnull().sum())
print("Final dataset shape:", model_df.shape)

In [ ]:
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

print("Training records:", len(train))
print("Testing records:", len(test))

In [ ]:
features = ['hour', 'day', 'previous_login_count', 'rolling_avg']

X_train = train[features]
y_train = train['failed_login_count']

X_test = test[features]
y_test = test['failed_login_count']

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Model training completed successfully!")

In [ ]:
y_pred = model.predict(X_test)

print("Actual values:", y_test.values[:10])
print("Predicted values:", y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(y_test.values, label='Actual', alpha=0.8)
plt.plot(y_pred, label='Predicted', alpha=0.8)

plt.xlabel("Test Intervals")
plt.ylabel("Failed Login Attempts")
plt.title("Actual vs Predicted Login Attempts")
plt.legend()
plt.show()

In [ ]:
results = test.copy()

results['predicted'] = y_pred
results['residual'] = results['failed_login_count'] - results['predicted']

print(results.head())